In [ ]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random, os
from datetime import datetime, timedelta
from google.colab import files

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
fake = Faker('en_US')
Faker.seed(RANDOM_SEED)
NUM_RECORDS = 50_000
print("✅ Ready")

✅ Ready


In [ ]:
CATEGORIES = ['Fast Food','Pizza','Sushi','Indian','Mexican','Chinese',
              'Burgers','Grocery','Pharmacy','Bakery']
CITIES = ['New York','Los Angeles','Chicago','Houston','Phoenix',
          'Philadelphia','San Antonio','San Diego','Dallas','Austin']

merchants = []
for i in range(1, 51):
    merchants.append({
        'merchant_id':        f'M{str(i).zfill(3)}',
        'merchant_name':      fake.company(),
        'category':           random.choice(CATEGORIES),
        'city':               random.choice(CITIES),
        'sla_target_minutes': int(np.random.choice([30, 35, 40, 45, 60]))
    })

merchants_df = pd.DataFrame(merchants)
merchant_lookup = merchants_df.set_index('merchant_id').to_dict(orient='index')
print(f"✅ {len(merchants_df)} merchants created")
merchants_df.head()

✅ 50 merchants created


,merchant_id,merchant_name,category,city,sla_target_minutes
0,M001,"Rodriguez, Figueroa and Sanchez",Pizza,New York,45
1,M002,Doyle Ltd,Mexican,Houston,60
2,M003,"Mcclain, Miller and Henderson",Indian,Chicago,40
3,M004,Davis and Sons,Pizza,Dallas,60
4,M005,"Guzman, Hoffman and Baldwin",Pizza,Austin,60


In [ ]:
START = datetime(2024, 1, 1)
END   = datetime(2024, 12, 31)
STATUSES = ['delivered','cancelled','failed','refunded']
WEIGHTS  = [0.85, 0.08, 0.05, 0.02]
VAL_RANGES = {
    'Fast Food':(8,20),'Pizza':(12,35),'Sushi':(25,75),
    'Indian':(15,45),'Mexican':(10,30),'Chinese':(12,40),
    'Burgers':(8,25),'Grocery':(20,120),'Pharmacy':(15,80),'Bakery':(5,25)
}

merchant_ids   = merchants_df['merchant_id'].tolist()
weights        = np.random.dirichlet(np.ones(50) * 2)
assigned       = np.random.choice(merchant_ids, size=NUM_RECORDS, p=weights)
total_seconds  = int((END - START).total_seconds())

rows = []
for i in range(NUM_RECORDS):
    mid    = assigned[i]
    m      = merchant_lookup[mid]
    cat    = m['category']
    sla    = m['sla_target_minutes']
    ts     = START + timedelta(seconds=random.randint(0, total_seconds))
    status = random.choices(STATUSES, weights=WEIGHTS)[0]
    lo, hi = VAL_RANGES[cat]
    val    = round(np.random.uniform(lo, hi), 2)

    if status == 'delivered':
        base     = np.random.normal(sla * 0.85, sla * 0.25)
        actual   = round(max(5, base + np.random.randint(1,6) * 1.5), 1)
        breached = actual > sla
        variance = round(actual - sla, 1)
    else:
        actual = breached = variance = None

    rows.append({
        'order_id':                f'ORD{str(i+1).zfill(6)}',
        'merchant_id':             mid,
        'merchant_name':           m['merchant_name'],
        'merchant_category':       cat,
        'city':                    m['city'],
        'order_timestamp':         ts.strftime('%Y-%m-%d %H:%M:%S'),
        'order_date':              ts.strftime('%Y-%m-%d'),
        'order_hour':              ts.hour,
        'order_month':             ts.month,
        'order_status':            status,
        'order_value_usd':         val,
        'sla_target_minutes':      sla,
        'actual_delivery_minutes': actual,
        'sla_breached':            breached,
        'sla_variance_minutes':    variance,
        'tablet_age_years':        int(np.random.randint(1, 6)),
        'customer_id':             f'CUST{str(np.random.randint(1,200001)).zfill(6)}',
        'driver_id':               f'DRV{str(np.random.randint(1,5001)).zfill(4)}',
    })

    if (i + 1) % 10_000 == 0:
        print(f"  ✔ {i+1:,} orders done...")

orders_df = pd.DataFrame(rows)
print(f"\n✅ Done! {len(orders_df):,} rows × {len(orders_df.columns)} columns")

  ✔ 10,000 orders done...
  ✔ 20,000 orders done...
  ✔ 30,000 orders done...
  ✔ 40,000 orders done...
  ✔ 50,000 orders done...

✅ Done! 50,000 rows × 18 columns


In [ ]:
os.makedirs('/content/data', exist_ok=True)
orders_df.to_csv('/content/data/merchant_orders.csv', index=False)
merchants_df.to_csv('/content/data/merchant_master.csv', index=False)
print("✅ Files saved")

files.download('/content/data/merchant_orders.csv')
files.download('/content/data/merchant_master.csv')
print("✅ Downloads triggered — check your Downloads folder")

✅ Files saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloads triggered — check your Downloads folder


In [ ]:
files.download('/content/data/merchant_master.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install snowflake-connector-python -q
print("✅ Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.5 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incom

In [ ]:
import getpass

# This shows a password box — type your Snowflake password and press Enter
# It won't show on screen (like a terminal password prompt)
SNOWFLAKE_PASSWORD = getpass.getpass("Enter your Snowflake password: ")
print("✅ Password stored")

Enter your Snowflake password: ··········
✅ Password stored


In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    account  = 'mdwahlm-mjb44570',
    user     = 'DKANSARA1',
    password = SNOWFLAKE_PASSWORD,
    warehouse= 'COMPUTE_WH',
    role     = 'ACCOUNTADMIN'
)

cursor = conn.cursor()
print("✅ Connected to Snowflake!")

✅ Connected to Snowflake!


In [ ]:
cursor.execute("CREATE DATABASE IF NOT EXISTS MERCHANT_DB;")
cursor.execute("USE DATABASE MERCHANT_DB;")
cursor.execute("CREATE SCHEMA IF NOT EXISTS RAW;")
cursor.execute("USE SCHEMA RAW;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS ORDERS_RAW (
    order_id                STRING,
    merchant_id             STRING,
    merchant_name           STRING,
    merchant_category       STRING,
    city                    STRING,
    order_timestamp         TIMESTAMP,
    order_date              DATE,
    order_hour              INTEGER,
    order_month             INTEGER,
    order_status            STRING,
    order_value_usd         FLOAT,
    sla_target_minutes      INTEGER,
    actual_delivery_minutes FLOAT,
    sla_breached            BOOLEAN,
    sla_variance_minutes    FLOAT,
    tablet_age_years        INTEGER,
    customer_id             STRING,
    driver_id               STRING
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS MERCHANT_MASTER (
    merchant_id        STRING,
    merchant_name      STRING,
    category           STRING,
    city               STRING,
    sla_target_minutes INTEGER
);
""")

print("✅ Database MERCHANT_DB created")
print("✅ Schema RAW created")
print("✅ Tables ORDERS_RAW and MERCHANT_MASTER created")

✅ Database MERCHANT_DB created
✅ Schema RAW created
✅ Tables ORDERS_RAW and MERCHANT_MASTER created


In [ ]:
import os

for f in ["/merchant_master.csv", "/merchant_orders.csv"]:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024*1024)
        print(f"✅ Found {f}  ({size:.1f} MB)")
    else:
        print(f"❌ NOT FOUND: {f}  — upload it first!")

✅ Found /merchant_master.csv  (0.0 MB)
✅ Found /merchant_orders.csv  (6.7 MB)


In [ ]:
cursor.execute("USE DATABASE MERCHANT_DB;")
cursor.execute("USE SCHEMA RAW;")
cursor.execute("CREATE STAGE IF NOT EXISTS merchant_stage;")

cursor.execute("PUT file:///content/data/merchant_orders.csv @merchant_stage AUTO_COMPRESS=TRUE;")
print("✅ merchant_orders.csv staged")

cursor.execute("""
COPY INTO ORDERS_RAW
FROM @merchant_stage/merchant_orders.csv.gz
FILE_FORMAT = (
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1
    NULL_IF = ('', 'None', 'NULL')
)
ON_ERROR = 'CONTINUE';
""")
print("✅ Orders loaded into ORDERS_RAW")

cursor.execute("PUT file:///content/data/merchant_master.csv @merchant_stage AUTO_COMPRESS=TRUE;")
print("✅ merchant_master.csv staged")

cursor.execute("""
COPY INTO MERCHANT_MASTER
FROM @merchant_stage/merchant_master.csv.gz
FILE_FORMAT = (
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1
    NULL_IF = ('', 'None', 'NULL')
)
ON_ERROR = 'CONTINUE';
""")
print("✅ Merchants loaded into MERCHANT_MASTER")

✅ merchant_orders.csv staged
✅ Orders loaded into ORDERS_RAW
✅ merchant_master.csv staged
✅ Merchants loaded into MERCHANT_MASTER


In [ ]:
cursor.execute("SELECT COUNT(*) FROM ORDERS_RAW;")
orders_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM MERCHANT_MASTER;")
merchants_count = cursor.fetchone()[0]

cursor.execute("""
    SELECT order_status, COUNT(*) as cnt
    FROM ORDERS_RAW
    GROUP BY order_status
    ORDER BY cnt DESC;
""")
status_breakdown = cursor.fetchall()

print(f"✅ ORDERS_RAW      → {orders_count:,} rows  (expected 50,000)")
print(f"✅ MERCHANT_MASTER → {merchants_count:,} rows  (expected 50)")
print(f"\nOrder status breakdown:")
for row in status_breakdown:
    print(f"   {row[0]:<15} {row[1]:,}")

conn.close()
print("\n✅ Connection closed — Step 2 complete!")

✅ ORDERS_RAW      → 50,000 rows  (expected 50,000)
✅ MERCHANT_MASTER → 50 rows  (expected 50)

Order status breakdown:
   delivered       42,479
   cancelled       4,009
   failed          2,509
   refunded        1,003

✅ Connection closed — Step 2 complete!
